# compact 特征 + czsc 笔 / 中枢研究

在 `calc_compact_features.ipynb` 的基础上，继续加入：

- **czsc 笔**：`CZSC.bi_list`（端点 `fx_a` / `fx_b`、方向、涨跌幅）；
- **缠论中枢**：`analysis.pivot` 的 `calc_pivots` / `plot_pivots`（`ZD/ZG/GG/DD`
  箱体、起止时间、方向、三买/三卖）—— 仅用于画图；
- **中枢计数 / 样品熵**：review 已落库的 `ZenPivotCnt` / `se`，attach 读取，
  **不在 notebook 重算**（D68）。

⚠️ czsc 0.7.10 的 `CZSC` 只暴露**笔**（`bi_list`），没有独立的“段”（`seg_list`）；
中枢里的 `zn` 是 Z 走势段，可在中枢表里查看。本 notebook **不伪造“段”**。

⚠️ 只读：不落库、不写水位、不做实时合并。默认取最近交易日。


In [ ]:
# 允许从 cookbooks/ 直接打开：向上找到含 GolemQ/__init__.py 的仓库根并加进 sys.path
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.isfile(os.path.join(_here, 'GolemQ', '__init__.py')):
        if _here not in sys.path:
            sys.path.insert(0, _here)
        break
    _parent = os.path.dirname(_here)
    if _parent == _here:
        break
    _here = _parent

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from GolemQ.core.constants import AKA, FEATURES as FTR, FIELD
from GolemQ.models.alias import PFL
from GolemQ.pipeline.compact import CompactPipeline

# 中文：与 4.4 老树 `utils/gfx.py` 同口径（Windows 自带微软雅黑）
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 80)
pd.set_option('expand_frame_repr', False)

In [ ]:
# 研究参数
code = '000711'
offset = ''          # '' = 最近交易日；或 'YYYY-MM-DD'

# czsc 最大笔数；与 review 的 `CZSC_MAX_BI` 一致
max_bi_count = 512


In [ ]:
import time

_t0 = time.perf_counter()
ohlc, features, display_name = CompactPipeline.calc_features(
    code, offset=offset, verbose=False, ret_meta=True)

print(f'{code} {display_name}  {len(features)} bars  {len(features.columns)} cols')
print(f'[timing] calc_compact_features: {time.perf_counter() - _t0:.2f}s')
features.tail(10)[[
    AKA.CLOSE, FIELD.CCI_NORM, FIELD.RSI, FIELD.MAPOWER30_MAJOR,
    FIELD.STOCK_SCORE_M15_NORM, FIELD.ATR_Stopline, PFL.STOCK_PORTFOLIO_RANK,
]]


## 附加 reality 特征（只读）

把 review/coldstart 与 ChartScan 的列接到 `features` 上（实现已收口到 `GolemQ/analysis/reality_attach.py`）：

- `bstStgBf`（`FLD.BOOTSTRAP_STAGE_BEFORE` 的落库名）← `stock_reality_feat_60min`；
- `_REAL` 系列（`zDLagMajR` / `poly9MajR` / `m9tMajR` / `rTrePrcMajR` …）← `stock_reality_feat_day`；
- ChartScan 三条时间线：`ClsMaj`（day）/ `Class`（60min）/ `ClsMin`（15min）。

⚠️ day 列按**因果**映射（盘中用上一交易日、15:00 用当日）；15min 只取**当前 60min bar 之前最后一个已完成**的 15min 值。

**K 线副坐标轴的两条周线归一曲线**：

- `POLYNOMIAL9_NORM_WEEKLY_REAL`（`poly9NorWekR`）—— 已由 review 落在 day 表；
- `DEA_NORM_WEK`（老树 `FLD.DEA_NORM_WEEKLY`）—— 新 review 契约**没有**这列，
  由日线 → 周线现算（周线 DEA → mapower 归一 0..1，按「上一已完成周」映射）。

- `COLDSTART_COLUMNS` 是**日派生连续量**：按日最后值 + **上一可用交易日**映射，
  周末/节后首日盘中不会再整列变空；
- 若 review 尚未写当日行，相关列保持 NaN；需在 review 侧补齐（不在 notebook 重算，D68）。

⚠️ 仍可能为 NaN 的列（不是 26 日映射问题）：

- `m9t` / `m9tLag`（compact 60min 版）本身就是稀疏序列（magic nine 在转向处写 NaN），
  且与 review 的同名列**同名** —— cell 5 不覆盖 compact 列；
- ChartScan 的 `Cls*` / `Conf*` / `Coord*` 是事件型稀疏数据，26 日没有对应事件就保持 NaN；
- `rStkPfl*` 是 rank 列，未满足 rank 条件时为 NaN（10-22/23 也一样）。

- **Prophet 四条/六条连续曲线**（`prophetLb`/`Ub`/`prophetLbMaj`/`UbMaj` …）：attach 末尾对
  缺口做 **ffill**（先补 15:00 真值再填，不会覆盖当日值），绘图不再出现肉眼断线。

- `se`（日收盘样本熵）/ `ZenPivotCnt`（已确认中枢数）也由 review
  滚动计算并落库；attach 精确 60min join 读取，**不在 notebook 重算**。


In [ ]:
# 附加 reality / coldstart / ChartScan 时间线（实现在 analysis/reality_attach.py）
import time
from GolemQ.analysis.reality_attach import DEA_NORM_WEEKLY, attach_reality_feats

_t0 = time.perf_counter()
_ts_all = pd.DatetimeIndex(pd.to_datetime(features.index.get_level_values(0)))
_, _day_ohlc, _ = CompactPipeline._fetch_frames(code, pd.Timestamp(_ts_all.max()))
features, loaded = attach_reality_feats(
    features, code,
    market_type=CompactPipeline.market_type_of(code),
    day_ohlc=_day_ohlc,
    verbose=False,
)
print('[attach] {}  ({}s)'.format(
    {k: len(v) for k, v in loaded.items()}, time.perf_counter() - _t0))
_show_cols = list(dict.fromkeys(c for cols in loaded.values() for c in cols))
features[_show_cols].tail(10)


In [ ]:
# 价格 K 线（mplfinance 蜡烛）+ ATR 通道 + 线性回归 + MA90
# 副坐标轴：周线归一曲线 POLYNOMIAL9_NORM_WEEKLY_REAL / DEA_NORM_WEEKLY（0..1）
import mplfinance as mpf
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

window = 1920
f = features.tail(window)
dates = pd.to_datetime(f.index.get_level_values(0))
x = np.arange(len(f))


def _time_axis(ax, dates, n=8):
    step = max(1, len(dates) // n)
    idx = np.arange(0, len(dates), step)
    ax.set_xticks(idx)
    ax.set_xticklabels([pd.Timestamp(d).strftime('%m-%d %H:%M')
                        for d in dates[idx]], fontsize=8)


# mplfinance 要 DatetimeIndex + 首字母大写的 OHLC 列
# 保留完整 60min 帧给下一格的 CZSC（画箱体/笔用；特征值不在这里重算）
raw_ohlc = ohlc
ohlc = f[[AKA.OPEN, AKA.HIGH, AKA.LOW, AKA.CLOSE]].copy()
ohlc.columns = ['Open', 'High', 'Low', 'Close']
ohlc.index = pd.DatetimeIndex(dates)

# ⚠️ mplfinance 会用 style 的 rc 覆盖全局 rcParams —— 中文字体必须
# 同时塞进 style，否则标题/刻度会退回 DejaVu Sans（豆腐块）。
ZH_RC = {
    'font.family': 'sans-serif',
    'font.sans-serif': ['Microsoft YaHei', 'SimHei', 'DejaVu Sans'],
    'axes.unicode_minus': False,
}
plt.rcParams.update(ZH_RC)

mc = mpf.make_marketcolors(up='#d62728', down='#2ca02c',
                           edge='inherit', wick='inherit')
style = mpf.make_mpf_style(marketcolors=mc, gridstyle='--',
                           gridcolor='#dddddd', rc=ZH_RC)

addplots = [
    mpf.make_addplot(f[FIELD.LINEAREG_PRICE].to_numpy(),
                     color='orange', linestyle='-.', width=1),
    mpf.make_addplot(f[AKA.CLOSE].rolling(90).mean().to_numpy(),
                     color='magenta', linestyle='--', width=1),
    mpf.make_addplot(f[FIELD.ATR_Stopline_PRICE].to_numpy(),
                     color='green', width=1),
]

# 副坐标轴上的两条周线归一曲线（缺列 / 全 NaN 时自动跳过）
WEEKLY_CURVES = [
    (FTR.POLYNOMIAL9_NORM_WEEKLY_REAL, 'tab:cyan',
     'POLYNOMIAL9_NORM_WEEKLY_REAL'),
    (DEA_NORM_WEEKLY, 'tab:brown', 'DEA_NORM_WEEKLY'),
]
for col, color, label in WEEKLY_CURVES:
    if col in f.columns and f[col].notna().any():
        addplots.append(mpf.make_addplot(
            f[col].to_numpy(), color=color, width=1.2,
            secondary_y=True, label=label))

fig, axes = mpf.plot(
    ohlc, type='candle', style=style, addplot=addplots,
    returnfig=True, figsize=(16, 7), warn_too_much_data=100000,
    xrotation=0, ylabel='price',
    title=f'{code} {display_name} — 60min OHLC / ATR band / LINEAREG',
)
ax = axes[0]
ax.fill_between(x, f[FIELD.ATR_LB].to_numpy(), f[FIELD.ATR_UB].to_numpy(),
                color='steelblue', alpha=0.12, zorder=0)

handles = [
    Line2D([0], [0], color='orange', ls='-.', label='LINEAREG_PRICE'),
    Line2D([0], [0], color='magenta', ls='--', label='MA90'),
    Line2D([0], [0], color='green', label='ATR_Stopline_PRICE'),
    Patch(facecolor='steelblue', alpha=0.2, label='ATR band'),
]
# 把副坐标轴的两条曲线并进同一个图例；只有真的加了副轴曲线才动 axes[-1]
if len(axes) > 1:
    sec = axes[-1]
    _sec_h, _sec_l = sec.get_legend_handles_labels()
    if _sec_h:
        sec.set_ylabel('weekly norm (0–1)', fontsize=9)
        sec.set_ylim(-0.05, 1.05)
        sec.tick_params(axis='y', labelsize=8)
        if sec.get_legend() is not None:
            sec.get_legend().remove()
        handles += _sec_h
ax.legend(handles=handles, loc='upper left', fontsize=9)
_time_axis(ax, dates)
# mplfinance 自己管布局；tight_layout 会告警，别加
plt.show()

## K 线 + 笔 + 中枢 + FBPROPHET

- 蓝色折线 = czsc 笔（`bi_list`，端点是小圆点）；
- 青色箱体 = 中枢 `[ZD, ZG]`（自 `known_dt` 成立起画，因果口径）；
- 虚线台阶 = 中枢的 `GG / DD`（只随新分型点刷新，不回填未来极值）；
- 紫色粗线 = 中枢的 `zn`（Z 走势段，按 high/low 方向绘制）；
- **主坐标轴**：`FBPROPHET_LB_MAJOR` / `FBPROPHET_UB_MAJOR`（`fetch_reality_feat`
  已把落库的比值乘回 close，画的是价格量纲的上下轨）；
- **副坐标轴（0–1）**：`DEA_NORM_WEEKLY` / `POLYNOMIAL9_NORM_WEEKLY_REAL` /
  `FBPROPHET_CLEARANCE_NORM_MAJOR`。

⚠️ 箱体/笔是本地 `analysis.pivot` 的**可视化**；`ZenPivotCnt` / `se` 用 review
落库值（`strict=True`、滚动日期计算），**不在 notebook 重算**。

⚠️ czsc 0.7.10 没有独立的“段”（`seg_list`）—— 紫色粗线是中枢的 `zn`
（Z 走势段，由 `calc_pivots` 的 `zn` 字段给出），**不是** czsc 的段对象；本 notebook 不伪造段。


In [ ]:
# K 线 + 笔/中枢：本地 CZSC/calc_pivots 仅用于**画箱体/笔**（review 未落库几何）。
# ⚠️ ZenPivotCnt / se 用 review 落库值（见 pivot-count / indicators），不在这里重算。
import time
from czsc import CZSC
from czsc.objects import RawBar, Freq
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from GolemQ.analysis.pivot import bi_confirm_map, calc_pivots, plot_pivots

_t0 = time.perf_counter()
bars = []
for i, (idx, row) in enumerate(raw_ohlc.iterrows()):
    dt = idx[0] if isinstance(idx, tuple) else idx
    bars.append(RawBar(
        symbol=code, dt=dt, id=i, freq=Freq.F60,
        open=float(row[AKA.OPEN]), close=float(row[AKA.CLOSE]),
        high=float(row[AKA.HIGH]), low=float(row[AKA.LOW]),
        vol=float(row.get(AKA.VOLUME, 0) or 0),
        amount=float(row.get('amount', 0) or 0)))
czsc = CZSC(bars, max_bi_count=max_bi_count, verbose=False)
conf_map = bi_confirm_map(bars, max_bi_count=max_bi_count)
pivots = calc_pivots(bi_list=czsc.bi_list, conf_map=conf_map, strict=True)
print(f'[czsc] bars={len(bars)} bi={len(czsc.bi_list)} '
      f'(box/bi visualization only; count/entropy from review)')

_pos = {pd.Timestamp(d): i for i, d in enumerate(dates)}


def _dt_to_x(dt):
    dt = pd.Timestamp(dt)
    if dt in _pos:
        return _pos[dt]
    return int(min(max(dates.searchsorted(dt), 0), len(dates) - 1))


addplots = []
# 主坐标轴：FBPROPHET 上下轨（fetch_reality_feat 已把比值乘回 close）
for col, color in ((FIELD.FBPROPHET_LB_MAJOR, 'tab:blue'),
                   (FIELD.FBPROPHET_UB_MAJOR, 'tab:red')):
    if col in f.columns and f[col].notna().any():
        addplots.append(mpf.make_addplot(
            f[col].to_numpy(), color=color, width=1.0, linestyle='--'))

# 副坐标轴：三条 0–1 归一曲线
SECONDARY_CURVES = [
    (DEA_NORM_WEEKLY, 'tab:brown', 'DEA_NORM_WEEKLY'),
    (FTR.POLYNOMIAL9_NORM_WEEKLY_REAL, 'tab:cyan',
     'POLYNOMIAL9_NORM_WEEKLY_REAL'),
    (FIELD.FBPROPHET_CLEARANCE_NORM_MAJOR, 'tab:olive',
     'FBPROPHET_CLEARANCE_NORM_MAJOR'),
]
for col, color, label in SECONDARY_CURVES:
    if col in f.columns and f[col].notna().any():
        addplots.append(mpf.make_addplot(
            f[col].to_numpy(), color=color, width=1.2,
            secondary_y=True, label=label))

fig, axes = mpf.plot(
    ohlc, type='candle', style=style, addplot=addplots or None,
    returnfig=True, figsize=(16, 7), warn_too_much_data=100000,
    xrotation=0, ylabel='price',
    title=f'{code} {display_name} — 60min 笔 / 中枢 / FBPROPHET',
)
ax = axes[0]

# 笔：与窗口有交集的才画（跨界笔的 x 会被 _dt_to_x 夹到窗口边缘）
for b in czsc.bi_list:
    if pd.Timestamp(b.fx_b.dt) < dates[0] or pd.Timestamp(b.fx_a.dt) > dates[-1]:
        continue
    x0, x1 = _dt_to_x(b.fx_a.dt), _dt_to_x(b.fx_b.dt)
    ax.plot([x0, x1], [b.fx_a.fx, b.fx_b.fx],
            color='royalblue', lw=1.0, alpha=0.9, zorder=3)
    ax.scatter([x0, x1], [b.fx_a.fx, b.fx_b.fx],
               s=8, color='royalblue', alpha=0.9, zorder=3)

# 中枢：只画与窗口有交集的；strict=True ⇒ 只画已确认的部分
_win_end = dates[-1]
pivots_win = [
    p for p in pivots
    if pd.Timestamp(p['end_dt'] or _win_end) >= dates[0]
    and pd.Timestamp(p['start_dt']) <= _win_end
]
# 中枢内的 Z 走势段（zn）：按方向从 high 画到 low —— 这是 czsc 0.7.10 里
# 唯一可用的“段”结构（不是 seg_list；本 notebook 不伪造 czsc 的段）
_zn_drawn = False
for p in pivots_win:
    for z in (p.get('zn') or []):
        if not (z.get('start_dt') and z.get('end_dt')):
            continue
        hi, lo = z.get('high'), z.get('low')
        if hi is None or lo is None:
            continue
        x0, x1 = _dt_to_x(z['start_dt']), _dt_to_x(z['end_dt'])
        if str(z.get('direction')).lower().startswith('down'):
            y0, y1 = hi, lo
        else:
            y0, y1 = lo, hi
        ax.plot([x0, x1], [y0, y1], color='purple', lw=2.0, alpha=0.45,
                zorder=2.5)
        _zn_drawn = True

# 研究图用 strict=False：完整显示历史箱体（含确认时间晚于 end_dt 的最后一段）；
# 特征/计数仍用 strict=True 的 known_dt 口径。
plot_pivots(ax, pivots_win, x_mapper=_dt_to_x, x_last=len(f) - 1,
            strict=False, label=True, fontsize=7, box_alpha=0.20)

handles = [
    Line2D([0], [0], color='royalblue', lw=1.2, label='czsc 笔'),
    Patch(facecolor='cyan', alpha=0.2, edgecolor='steelblue',
          label='中枢 ZD–ZG'),
]
if _zn_drawn:
    handles.append(Line2D([0], [0], color='purple', lw=2.0, alpha=0.55,
                          label='Z 走势段 (zn)'))
if FIELD.FBPROPHET_LB_MAJOR in f.columns and f[FIELD.FBPROPHET_LB_MAJOR].notna().any():
    handles.append(Line2D([0], [0], color='tab:blue', ls='--',
                          label='FBPROPHET_LB_MAJOR'))
if FIELD.FBPROPHET_UB_MAJOR in f.columns and f[FIELD.FBPROPHET_UB_MAJOR].notna().any():
    handles.append(Line2D([0], [0], color='tab:red', ls='--',
                          label='FBPROPHET_UB_MAJOR'))
if len(axes) > 1:
    sec = axes[-1]
    _sec_h, _sec_l = sec.get_legend_handles_labels()
    if _sec_h:
        sec.set_ylabel('norm (0–1)', fontsize=9)
        sec.set_ylim(-0.05, 1.05)
        sec.tick_params(axis='y', labelsize=8)
        if sec.get_legend() is not None:
            sec.get_legend().remove()
        handles += _sec_h
ax.legend(handles=handles, loc='upper left', fontsize=8)
_time_axis(ax, dates)
plt.show()


In [ ]:
# ZenPivotCnt 由 review 落库（滚动日期计算）—— 这里只读取并展示，不重算
counts = (f['ZenPivotCnt'] if 'ZenPivotCnt' in f.columns
          else pd.Series(np.nan, index=f.index))
_valid = counts.dropna()
if len(_valid):
    print(f'ZenPivotCnt: first={_valid.iloc[0]:.0f} @ {_valid.index[0]}, '
          f'last={_valid.iloc[-1]:.0f} @ {_valid.index[-1]}, n={len(_valid)}')
    if len(_valid) < len(counts):
        print(f'⚠️ 只有 {len(_valid)}/{len(counts)} 行有 ZenPivotCnt '
              f'（review 未回填本窗口的历史行；需 review --force/--from）')
else:
    print('review 未写 ZenPivotCnt（本窗口）')

_extra = [c for c in ('ZenPivotCnt', 'se') if c in f.columns]
if _extra:
    print(f[_extra].dropna(how='all').tail(5).to_string())

fig, ax = plt.subplots(figsize=(16, 3))
ax.step(x, counts, where='post', color='steelblue', lw=1.2)
ax.fill_between(x, 0, counts, step='post', color='steelblue', alpha=0.12)
ax.set_ylabel('ZenPivotCnt', fontsize=9)
ax.set_title(f'{code} ZenPivotCnt (review)', fontsize=10)
ax.grid(alpha=0.25)
_time_axis(ax, dates)
plt.tight_layout()
plt.show()


In [ ]:
# 指标面板：CCI / RSI / MAPOWER30 / 股票评分
panels = [
    (FIELD.CCI_NORM, 'CCI_NORM', 'tab:blue'),
    (FIELD.RSI, 'RSI', 'tab:orange'),
    (FIELD.MAPOWER30_MAJOR, 'MAPOWER30_MAJOR', 'tab:purple'),
    (FIELD.STOCK_SCORE_M15_NORM, 'STOCK_SCORE_M15_NORM', 'tab:red'),
]
if AKA.SAMPLE_ENTROPY in f.columns:          # review 落库的 se
    panels.append((AKA.SAMPLE_ENTROPY, 'SE (sample entropy)', 'tab:brown'))
fig, axes = plt.subplots(len(panels), 1, figsize=(16, 11), sharex=True)
for ax, (col, label, color) in zip(axes, panels):
    ax.plot(x, f[col], color=color, lw=1)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(alpha=0.25)
_time_axis(axes[-1], dates)
axes[-1].set_xlabel('time')
fig.suptitle(f'{code} compact indicators', y=0.995)
plt.tight_layout()
plt.show()


In [ ]:
# rank / 收益
fig, ax = plt.subplots(figsize=(16, 4.5))
ax.plot(x, f[PFL.STOCK_PORTFOLIO_RANK], color='tab:blue', lw=1,
        label=PFL.STOCK_PORTFOLIO_RANK)
ax.plot(x, f[PFL.STOCK_PORTFOLIO_RANK_NEGATIVE], color='tab:red', lw=1,
        alpha=0.7, label=PFL.STOCK_PORTFOLIO_RANK_NEGATIVE)
ax.set_title(f'{code} portfolio rank (60min)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.25)
_time_axis(ax, dates)
plt.tight_layout()
plt.show()
